## CNN Sandbox Developer

This notebook serves to test updates, and make changes without affecting the src branch.

In [21]:
import pandas as pd
from datetime import timedelta, datetime
import bs4
import requests
import random
from random import randint
import time
import re
from src.web_scrapers.static_info import user_agents
from src.text_processing.text_clean import clean_text

In [8]:
start_date = pd.to_datetime('2026-08-03')
end_date = pd.to_datetime('2026-08-04')

month = start_date.month
year = start_date.year

print(month)
print(year)

8
2026


### Article Links

In [10]:
cnn_date, cnn_href, cnn_title = [], [], []

base_url = 'https://www.cnn.com/sitemap/article/'

month = start_date.month
year = start_date.year

headers = {'User-Agent': random.choice(user_agents)}

url = f"{base_url}{year}/{month}"

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = bs4.BeautifulSoup(response.content, 'html.parser')

    articles = []
    items = soup.select('ul.sitemap__list li.sitemap__list-item')

    for item in items:
        # Extract date
        date_elem = item.select_one('span.sitemap__list-item__date')
        date = date_elem.text.strip() if date_elem else None

        # Extract link and title
        link_elem = item.select_one('a.sitemap__list-item__link')
        href = link_elem.get('href') if link_elem else None
        title = link_elem.text.strip() if link_elem else None

        cnn_date.append(date)
        cnn_href.append(href)
        cnn_title.append(title)

else:
    print(f"Status code: {response.status_code}")

time.sleep(randint(2, 7))

links = pd.DataFrame({'date': cnn_date, "source": "cnn", 'href': cnn_href, 'title': cnn_title})

exclude_pattern = r'/cnn-underscored/|/style/|/entertainment/|/games/|media\.cnn\.com'
links = links[~links['href'].str.contains(exclude_pattern, na=False, regex=True)]

links = links.sort_values(by='date', na_position='last')
links = links.drop_duplicates(subset=['href'], keep='first').reset_index(drop=True)
links['date'] = pd.to_datetime(links['date'])
links[(links['date'] >= start_date) & (links['date'] <= end_date)]

In [ ]:
links.head()

### Article Contents

In [ ]:
article_contents, article_hrefs = [], []

headers = {'User-Agent': random.choice(user_agents)}

for hrf in links['href']:
    url = hrf

    response = requests.get(url, allow_redirects=False, headers=headers)

    if response.status_code == 200:
        soup = bs4.BeautifulSoup(response.content, "html.parser")
        data = soup.find("div", {"class": "article__content-container"})
        try:
            tag = [x.text for x in data.select("p")]
            tag_content = " ".join(tag)

            article_contents.append(tag_content)
            article_hrefs.append(hrf)

        except AttributeError:
            continue

    time.sleep(randint(19, 39))

cnn_contents = pd.DataFrame({"href": article_hrefs, "content": article_contents})
cnn = pd.merge(left=links, right=cnn_contents, left_on='href', right_on='href')

In [22]:
links['title'] = links['title'].apply(clean_text)
links.head()

,date,source,href,title
0,2026-08-01,cnn,https://www.cnn.com/2026/08/01/europe/spain-wi...,No species left behind: how Spain saved its an...
1,2026-08-01,cnn,https://www.cnn.com/2026/08/01/politics/the-tr...,Trump's Reflecting Pool debacle takes a predic...
2,2026-08-01,cnn,https://www.cnn.com/2026/08/01/weather/santa-r...,Massive die-off predicted for exceptionally ra...
3,2026-08-01,cnn,https://www.cnn.com/2026/08/01/us/savannah-gut...,'Our hearts are in ruins': Savannah Guthrie's ...
4,2026-08-01,cnn,https://www.cnn.com/2026/08/01/politics/pulte-...,Acting director of national intelligence says ...
